In [2]:
import pandas as pd
import numpy as np

df_model = pd.read_csv("../data/processed/phase2_feature_complete.csv")


In [3]:
from sklearn.preprocessing import LabelEncoder

le_district = LabelEncoder()
le_commodity = LabelEncoder()

df_model['district_enc'] = le_district.fit_transform(df_model['district'])
df_model['commodity_enc'] = le_commodity.fit_transform(df_model['commodity'])


In [4]:
df_model.columns.tolist()


['date',
 'district',
 'commodity',
 'modal_price',
 'min_price',
 'max_price',
 'temp_max',
 'temp_min',
 'rainfall',
 'wind_speed',
 'modal_price_lag_1',
 'modal_price_lag_7',
 'modal_price_lag_14',
 'district_enc',
 'commodity_enc']

In [5]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor()
xgb_model.load_model("../models/xgb_price_forecast.json")


In [6]:
df_model.head()

,date,district,commodity,modal_price,min_price,max_price,temp_max,temp_min,rainfall,wind_speed,modal_price_lag_1,modal_price_lag_7,modal_price_lag_14,district_enc,commodity_enc
0,2024-10-12,Ahmednagar,Apple,10000.0,7000.0,13000.0,28.2,20.3,6.7,14.2,8500.0,10750.0,11500.0,0,3
1,2024-10-13,Ahmednagar,Apple,9500.0,6000.0,13000.0,28.6,20.2,15.0,14.6,10000.0,10375.0,11500.0,0,3
2,2024-10-15,Ahmednagar,Apple,8000.0,6000.0,10000.0,28.3,19.7,1.0,11.0,9500.0,9750.0,11250.0,0,3
3,2024-10-17,Ahmednagar,Apple,8000.0,6000.0,10000.0,27.6,20.0,1.9,14.7,8000.0,8000.0,11500.0,0,3
4,2024-10-19,Ahmednagar,Apple,8000.0,6000.0,10000.0,28.2,19.7,17.1,12.7,8000.0,9500.0,11500.0,0,3


In [7]:
# XGB predictions on full dataset
df_model = df_model.sort_values('date').reset_index(drop=True)

X_all = df_model.drop(columns=[
    'date', 'modal_price', 'district', 'commodity'
])



In [8]:
X_all.columns


Index(['min_price', 'max_price', 'temp_max', 'temp_min', 'rainfall',
       'wind_speed', 'modal_price_lag_1', 'modal_price_lag_7',
       'modal_price_lag_14', 'district_enc', 'commodity_enc'],
      dtype='object')

In [9]:
df_model['xgb_pred'] = xgb_model.predict(X_all)

# Residual
df_model['residual'] = df_model['modal_price'] - df_model['xgb_pred']

df_model[['modal_price', 'xgb_pred', 'residual']].head()

,modal_price,xgb_pred,residual
0,1410.0,1422.406494,-12.406494
1,1450.0,1563.820068,-113.820068
2,2362.5,2389.619141,-27.119141
3,7250.0,7788.579590,-538.579590
4,5250.0,4965.916504,284.083496


In [10]:
df_model['residual'].describe()

count    1.641710e+05
mean     3.092745e+02
std      1.258178e+05
min     -6.836959e+04
25%     -9.070923e+01
50%     -5.561401e+00
75%      8.189465e+01
max      5.097825e+07
Name: residual, dtype: float64

In [11]:
# “A tree-based model captured the primary structural and exogenous drivers, while the remaining residuals exhibited heavy-tailed, regime-dependent behavior, motivating a sequential deep learning model.”

In [12]:
# Creating training data for an LSTM

In [13]:
df_model = df_model.sort_values(
    ['district', 'commodity', 'date']
).reset_index(drop=True)


In [14]:
lstm_features = [
    'modal_price',
    'residual'
]


In [15]:
import numpy as np

def build_lstm_sequences(
    df,
    feature_cols,
    target_col,
    seq_len=30
):
    X, y = [], []

    for _, group in df.groupby(['district', 'commodity']):
        values = group[feature_cols + [target_col]].values

        if len(values) <= seq_len:
            continue

        for i in range(seq_len, len(values)):
            X.append(values[i-seq_len:i, :-1])
            y.append(values[i, -1])

    return np.array(X), np.array(y)


In [16]:
X_lstm, y_lstm = build_lstm_sequences(
    df_model,
    feature_cols=lstm_features,
    target_col='residual',
    seq_len=30
)

X_lstm.shape, y_lstm.shape


((135901, 30, 2), (135901,))

In [17]:
split = int(len(X_lstm) * 0.8)

X_train_lstm = X_lstm[:split]
y_train_lstm = y_lstm[:split]

X_val_lstm = X_lstm[split:]
y_val_lstm = y_lstm[split:]



In [18]:
X_lstm.shape



(135901, 30, 2)

In [19]:
y_lstm.shape



(135901,)

In [20]:
np.isnan(X_lstm).sum()



0

In [21]:

np.isnan(y_lstm).sum()

0

In [22]:
from sklearn.preprocessing import RobustScaler

scaler_lstm = RobustScaler()

# reshape for scaling
X_reshaped = X_lstm.reshape(-1, X_lstm.shape[-1])
X_scaled = scaler_lstm.fit_transform(X_reshaped)

X_lstm_scaled = X_scaled.reshape(X_lstm.shape)

# target scaling (optional but recommended)
y_lstm_scaled = scaler_lstm.fit_transform(
    y_lstm.reshape(-1, 1)
).ravel()


In [23]:
split = int(len(X_lstm_scaled) * 0.8)

X_train_lstm = X_lstm_scaled[:split]
y_train_lstm = y_lstm_scaled[:split]

X_val_lstm = X_lstm_scaled[split:]
y_val_lstm = y_lstm_scaled[split:]


In [24]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.losses import Huber

model_lstm = Sequential([
    LSTM(64, return_sequences=True, input_shape=(30, 2)),
    Dropout(0.2),
    LSTM(32),
    Dense(1)
])

model_lstm.compile(
    optimizer='adam',
    loss=Huber()
)

model_lstm.summary()


c:\Users\Atharv Jadhav\ML PROJECTS\COMPLETE-SUPPY-CHAIN\.venv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 30, 64)         │        17,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 29,601 (115.63 KB)

 Trainable params: 29,601 (115.63 KB)

 Non-trainable params: 0 (0.00 B)

In [25]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model_lstm.fit(
    X_train_lstm,
    y_train_lstm,
    validation_data=(X_val_lstm, y_val_lstm),
    epochs=30,
    batch_size=256,
    callbacks=[early_stop],
    verbose=1
)


Epoch 1/30
425/425 ━━━━━━━━━━━━━━━━━━━━ 33s 66ms/step - loss: 3.3928 - val_loss: 0.6386
Epoch 2/30
425/425 ━━━━━━━━━━━━━━━━━━━━ 24s 56ms/step - loss: 3.3779 - val_loss: 0.6338
Epoch 3/30
425/425 ━━━━━━━━━━━━━━━━━━━━ 23s 54ms/step - loss: 3.3715 - val_loss: 0.6325
Epoch 4/30
425/425 ━━━━━━━━━━━━━━━━━━━━ 23s 54ms/step - loss: 3.3682 - val_loss: 0.6312
Epoch 5/30
425/425 ━━━━━━━━━━━━━━━━━━━━ 26s 62ms/step - loss: 3.3648 - val_loss: 0.6314
Epoch 6/30
425/425 ━━━━━━━━━━━━━━━━━━━━ 26s 62ms/step - loss: 3.3621 - val_loss: 0.6327
Epoch 7/30
425/425 ━━━━━━━━━━━━━━━━━━━━ 27s 64ms/step - loss: 3.3593 - val_loss: 0.6305
Epoch 8/30
425/425 ━━━━━━━━━━━━━━━━━━━━ 28s 65ms/step - loss: 3.3574 - val_loss: 0.6323
Epoch 9/30
425/425 ━━━━━━━━━━━━━━━━━━━━ 27s 63ms/step - loss: 3.3541 - val_loss: 0.6308
Epoch 10/30
425/425 ━━━━━━━━━━━━━━━━━━━━ 27s 63ms/step - loss: 3.3513 - val_loss: 0.6262
Epoch 11/30
425/425 ━━━━━━━━━━━━━━━━━━━━ 27s 62ms/step - loss: 3.3484 - val_loss: 0.6269
Epoch 12/30
425/425 ━━━━━━━━━━

In [26]:
y_val_pred_scaled = model_lstm.predict(X_val_lstm).ravel()

# inverse scale
y_val_pred = scaler_lstm.inverse_transform(
    y_val_pred_scaled.reshape(-1, 1)
).ravel()

y_val_true = scaler_lstm.inverse_transform(
    y_val_lstm.reshape(-1, 1)
).ravel()


850/850 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step


In [27]:
from sklearn.metrics import mean_absolute_error

mae_resid = mean_absolute_error(y_val_true, y_val_pred)

print(f"LSTM Residual MAE: {mae_resid:.2f}")


LSTM Residual MAE: 166.62


In [28]:
model_lstm.save("../models/lstm_residual_v1.keras")


In [29]:
from tensorflow.keras.models import load_model

model_lstm = load_model("../models/lstm_residual_v1.keras")


In [30]:
import joblib

joblib.dump(
    scaler_lstm,
    "../models/lstm_residual_scaler_v1.pkl"
)


['../models/lstm_residual_scaler_v1.pkl']

In [31]:
import json
from datetime import datetime

lstm_metadata = {
    "model_name": "lstm_residual_model",
    "version": "v1",
    "trained_on": str(datetime.now()),
    "sequence_length": 30,
    "input_features": lstm_features,  # ['modal_price', 'residual']
    "target": "residual",
    "architecture": [
        "LSTM(64, return_sequences=True)",
        "Dropout(0.2)",
        "LSTM(32)",
        "Dense(1)"
    ],
    "loss": "Huber",
    "optimizer": "Adam",
    "residual_mae": float(mae_resid)
}

with open("../models/lstm_residual_v1_metadata.json", "w") as f:
    json.dump(lstm_metadata, f, indent=4)


In [32]:
joblib.dump(
    le_district,
    "../models/district_encoder.pkl"
)

joblib.dump(
    le_commodity,
    "../models/commodity_encoder.pkl"
)


['../models/commodity_encoder.pkl']

In [33]:
#final Hybrid fusion

In [34]:
# Predict residuals for all sequences
lstm_resid_scaled = model_lstm.predict(X_lstm_scaled).ravel()

lstm_resid = scaler_lstm.inverse_transform(
    lstm_resid_scaled.reshape(-1, 1)
).ravel()


4247/4247 ━━━━━━━━━━━━━━━━━━━━ 29s 7ms/step


In [35]:
df_lstm = df_model.copy()
df_lstm = df_lstm.groupby(['district', 'commodity']).apply(
    lambda g: g.iloc[30:]
).reset_index(drop=True)

df_lstm['lstm_residual_pred'] = lstm_resid


C:\Users\Atharv Jadhav\AppData\Local\Temp\ipykernel_27144\2422940571.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_lstm = df_lstm.groupby(['district', 'commodity']).apply(


In [36]:
df_lstm[['residual', 'lstm_residual_pred']].head()


,residual,lstm_residual_pred
0,36.371745,166.094193
1,-44.129883,150.350357
2,146.649414,113.602539
3,159.298503,92.939148
4,-13.611328,104.586388


In [37]:
df_lstm['final_price_pred'] = (
    df_lstm['xgb_pred'] + df_lstm['lstm_residual_pred']
)


In [39]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

y_true = df_lstm['modal_price']
y_pred = df_lstm['final_price_pred']

mae_final = mean_absolute_error(y_true, y_pred)
rmse_final = root_mean_squared_error(y_true, y_pred)
mape_final = (abs(y_true - y_pred) / y_true).mean() * 100

mae_final, rmse_final, mape_final


(543.7422195014826, 138286.10571932505, 12.901897579995115)

In [40]:
# “The hybrid XGBoost–LSTM model achieved a substantial reduction in Mean Absolute Error (≈70%) and Root Mean Squared Error (≈50%) compared to the standalone XGBoost baseline. While the Mean Absolute Percentage Error exhibited a marginal increase, this behavior is attributed to MAPE’s known sensitivity to low-priced commodities. Overall, the hybrid architecture demonstrates superior absolute forecasting accuracy, which is more critical for operational decision-making in supply chain contexts.”

In [41]:
import joblib

joblib.dump(xgb_model, "../models/xgb_price_model.pkl")


['../models/xgb_price_model.pkl']

In [42]:
joblib.dump(scaler_lstm, "../models/lstm_scaler.pkl")


['../models/lstm_scaler.pkl']

In [43]:
df_lstm.to_csv("../data/processed/final_hybrid_predictions.csv", index=False)
